In [1]:
import ee
ee.Initialize()

/home/wmlegion/miniconda3/envs/agri_land_env/lib/python3.10/site-packages/google/api_core/_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
from geopy.geocoders import Nominatim

def get_location_name(lat, lon):
    # Initialize Nominatim geocoder (requires a custom user_agent string)
    geolocator = Nominatim(user_agent="agri_land_suitability_pipeline")

    try:
        # Perform reverse geocoding
        location = geolocator.reverse((lat, lon), language='en')

        if location and location.raw:
            address = location.raw.get('address', {})

            # Extract city/town/village and country safely
            city = (
                address.get('city') or 
                address.get('town') or 
                address.get('village') or 
                address.get('municipality') or 
                address.get('county') or 
                "Unknown City"
            )
            country = address.get('country', "Unknown Country")

            return {"city": city, "country": country}
        else:
            return {"city": "Not found", "country": "Not found"}

    except Exception as e:
        return {"error": str(e)}

# Test it with your coordinates (7.3297, -73.1867)
location_info = get_location_name(7.300921,-73.009794)
print("Location:", location_info)
#Location: {'city': 'Bundaberg', 'country': 'Australia'}(-24.8660, 152.3489)

Location: {'city': 'Matanza', 'country': 'Colombia'}


In [ ]:
from geopy.geocoders import Nominatim

def get_location_name(lat, lon):
    # Initialize Nominatim geocoder (requires a custom user_agent string)
    geolocator = Nominatim(user_agent="agri_land_suitability_pipeline")

    try:
        # Perform reverse geocoding
        location = geolocator.reverse((lat, lon), language='en')

        if location and location.raw:
            address = location.raw.get('address', {})

            # Extract city/town/village safely
            city = (
                address.get('city') or 
                address.get('town') or 
                address.get('village') or 
                address.get('municipality') or 
                address.get('county') or 
                "Unknown City"
            )
            
            # Extract state/province safely (handles different regional naming conventions)
            state = (
                address.get('state') or 
                address.get('province') or 
                address.get('region') or 
                address.get('state_district') or 
                "Unknown State"
            )
            
            country = address.get('country', "Unknown Country")

            return {"city": city, "state": state, "country": country}
        else:
            return {"city": "Not found", "state": "Not found", "country": "Not found"}

    except Exception as e:
        return {"error": str(e)}

# Test with your coordinates where city came up as Unknown City
# Finca Matanza 7.300921,-73.009794
location_info = get_location_name(-24.8660, 152.3489)
print("Location:", location_info)

Location: {'city': 'Bundaberg', 'state': 'Queensland', 'country': 'Australia'}


In [7]:
print(location_info['state'])

Queensland


In [3]:
import math
import time
import requests
from PIL import Image
from geopy.geocoders import Nominatim

# --- 1. Obtener el bounding box del state ---
def get_state_bbox(state_name, country="Colombia"):
    geolocator = Nominatim(user_agent="agri_land_suitability_pipeline")
    location = geolocator.geocode(f"{state_name}, {country}", exactly_one=True)
    if not location:
        raise ValueError("No se encontró el state")
    # boundingbox viene como [south, north, west, east] en strings
    south, north, west, east = map(float, location.raw['boundingbox'])
    return south, north, west, east

# --- 2. Conversión lat/lon <-> número de tile (fórmula estándar Web Mercator) ---
def latlon_to_tile(lat, lon, zoom):
    lat_rad = math.radians(lat)
    n = 2 ** zoom
    x = int((lon + 180.0) / 360.0 * n)
    y = int((1.0 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2.0 * n)
    return x, y

def tile_to_latlon(x, y, zoom):
    n = 2 ** zoom
    lon = x / n * 360.0 - 180.0
    lat_rad = math.atan(math.sinh(math.pi * (1 - 2 * y / n)))
    lat = math.degrees(lat_rad)
    return lat, lon

# --- 3. Descargar y pegar los tiles ---
def get_state_elevation_map(state_name, country="Colombia", zoom=10, out_path="state_map.png"):
    south, north, west, east = get_state_bbox(state_name, country)

    x_min, y_min = latlon_to_tile(north, west, zoom)  # esquina superior-izq
    x_max, y_max = latlon_to_tile(south, east, zoom)  # esquina inferior-der

    tile_size = 256
    width = (x_max - x_min + 1) * tile_size
    height = (y_max - y_min + 1) * tile_size
    canvas = Image.new("RGB", (width, height))

    headers = {"User-Agent": "agri_land_suitability_pipeline (tu_email@ejemplo.com)"}

    for x in range(x_min, x_max + 1):
        for y in range(y_min, y_max + 1):
            url = f"https://tile.opentopomap.org/{zoom}/{x}/{y}.png"
            resp = requests.get(url, headers=headers, timeout=10)
            if resp.status_code == 200:
                tile_img = Image.open(requests.compat.io.BytesIO(resp.content)) if False else None
            time.sleep(0.5)  # respeta el rate limit (~2 req/seg máx)
            try:
                from io import BytesIO
                tile_img = Image.open(BytesIO(resp.content))
                canvas.paste(tile_img, ((x - x_min) * tile_size, (y - y_min) * tile_size))
            except Exception as e:
                print(f"Tile {x},{y} falló: {e}")

    canvas.save(out_path)
    print(f"Mapa guardado en {out_path} ({width}x{height}px)")
    return out_path

# Prueba
get_state_elevation_map("Santander", "Colombia", zoom=9)


Mapa guardado en state_map.png (768x1024px)


'state_map.png'

In [ ]:
import math
import time
from io import BytesIO
import requests
from PIL import Image
from geopy.geocoders import Nominatim


# --- 1. Obtener el bounding box del state ---
def get_state_bbox(state_name, country="Colombia"):
    geolocator = Nominatim(user_agent="agri_land_suitability_pipeline")
    location = geolocator.geocode(f"{state_name}, {country}", exactly_one=True)
    if not location:
        raise ValueError(f"No se encontró el state '{state_name}, {country}' en Nominatim")
    print(f"[DEBUG] Nominatim encontró: {location.address}")
    south, north, west, east = map(float, location.raw['boundingbox'])
    print(f"[DEBUG] bbox -> south={south}, north={north}, west={west}, east={east}")
    return south, north, west, east


# --- 2. Conversión lat/lon <-> número de tile ---
def latlon_to_tile(lat, lon, zoom):
    lat_rad = math.radians(lat)
    n = 2 ** zoom
    x = int((lon + 180.0) / 360.0 * n)
    y = int((1.0 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2.0 * n)
    return x, y


# --- 3. Descargar y pegar los tiles (con debug) ---
def get_state_elevation_map(state_name, country="Colombia", zoom=9, out_path="state_map.png"):
    south, north, west, east = get_state_bbox(state_name, country)

    x_min, y_min = latlon_to_tile(north, west, zoom)
    x_max, y_max = latlon_to_tile(south, east, zoom)
    print(f"[DEBUG] tiles a descargar: x {x_min}-{x_max}, y {y_min}-{y_max} "
          f"({(x_max - x_min + 1) * (y_max - y_min + 1)} tiles)")

    tile_size = 256
    width = (x_max - x_min + 1) * tile_size
    height = (y_max - y_min + 1) * tile_size
    canvas = Image.new("RGB", (width, height))

    # OJO: pon un email/URL real tuyo aquí, no un placeholder falso.
    # Muchos servidores de tiles rechazan (403) User-Agents genéricos.
    headers = {"User-Agent": "agri_land_suitability_pipeline/1.0 (contacto: tu_correo_real@dominio.com)"}

    exitosos, fallidos = 0, 0
    for x in range(x_min, x_max + 1):
        for y in range(y_min, y_max + 1):
            url = f"https://tile.opentopomap.org/{zoom}/{x}/{y}.png"
            resp = requests.get(url, headers=headers, timeout=10)
            print(f"[DEBUG] tile {x},{y} -> status {resp.status_code}, "
                  f"{len(resp.content)} bytes, content-type={resp.headers.get('Content-Type')}")

            if resp.status_code != 200:
                print(f"[DEBUG]   contenido de la respuesta (primeros 200 chars): {resp.text[:200]}")
                fallidos += 1
                time.sleep(0.5)
                continue

            try:
                tile_img = Image.open(BytesIO(resp.content))
                canvas.paste(tile_img, ((x - x_min) * tile_size, (y - y_min) * tile_size))
                exitosos += 1
            except Exception as e:
                print(f"[DEBUG]   Image.open falló para tile {x},{y}: {e}")
                fallidos += 1

            time.sleep(0.5)  # respeta el rate limit (~2 req/seg máx)

    print(f"[DEBUG] tiles exitosos: {exitosos}, fallidos: {fallidos}")
    canvas.save(out_path)
    print(f"Mapa guardado en {out_path} ({width}x{height}px)")
    return out_path


if __name__ == "__main__":
    # Reemplaza esto con los valores reales de tu location_info
    # location_info = {"state": "Santander", "country": "Colombia"}
    get_state_elevation_map(location_info['state'], location_info['country'], zoom=9)

[DEBUG] Nominatim encontró: Queensland, Australia
[DEBUG] bbox -> south=-29.179266, north=-9.0880125, west=137.9946464, east=153.6116035
[DEBUG] tiles a descargar: x 452-474, y 268-299 (736 tiles)
[DEBUG] tile 452,268 -> status 200, 3290 bytes, content-type=image/png
[DEBUG] tile 452,269 -> status 200, 629 bytes, content-type=image/png
[DEBUG] tile 452,270 -> status 200, 756 bytes, content-type=image/png
[DEBUG] tile 452,271 -> status 200, 103 bytes, content-type=image/png
[DEBUG] tile 452,272 -> status 200, 103 bytes, content-type=image/png
[DEBUG] tile 452,273 -> status 200, 103 bytes, content-type=image/png
[DEBUG] tile 452,274 -> status 200, 103 bytes, content-type=image/png
[DEBUG] tile 452,275 -> status 200, 103 bytes, content-type=image/png
[DEBUG] tile 452,276 -> status 200, 103 bytes, content-type=image/png
[DEBUG] tile 452,277 -> status 200, 103 bytes, content-type=image/png
[DEBUG] tile 452,278 -> status 200, 1415 bytes, content-type=image/png
[DEBUG] tile 452,279 -> status 

In [13]:
print(location_info['state'])
print(location_info['country'])

Queensland
Australia


In [5]:
import os
import elevation
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
from geopy.geocoders import Nominatim


# --- 1. Obtener el bounding box del state (igual que antes) ---
def get_state_bbox(state_name, country="Colombia"):
    geolocator = Nominatim(user_agent="agri_land_suitability_pipeline")
    location = geolocator.geocode(f"{state_name}, {country}", exactly_one=True)
    if not location:
        raise ValueError(f"No se encontró el state '{state_name}, {country}' en Nominatim")
    south, north, west, east = map(float, location.raw['boundingbox'])
    print(f"[DEBUG] bbox -> south={south}, north={north}, west={west}, east={east}")
    return south, north, west, east


# --- 2. Descargar el DEM recortado al bbox (un solo GeoTIFF) ---
def get_state_dem(state_name, country="Colombia", out_path="state_dem.tif"):
    south, north, west, east = get_state_bbox(state_name, country)

    # elevation.clip espera (west, south, east, north)
    bounds = (west, south, east, north)
    print(f"[DEBUG] descargando DEM para bounds={bounds} ...")

    # max_download_tiles: la librería limita por defecto la cantidad de tiles SRTM
    # para evitar bulk-download accidental. Lo subimos a un tope razonable para
    # cubrir states grandes (Queensland necesitaría bastantes más que Santander).
    elevation.clip(bounds=bounds, output=os.path.abspath(out_path), max_download_tiles=50)
    elevation.clean()  # borra los tiles temporales intermedios, deja solo el resultado

    print(f"[DEBUG] DEM guardado en {out_path}")
    return out_path


# --- 3. Renderizar el DEM con hillshade (visual similar a OpenTopoMap) ---
def render_dem(dem_path, out_image="state_map.png"):
    with rasterio.open(dem_path) as src:
        elevation_data = src.read(1).astype(float)
        elevation_data[elevation_data == src.nodata] = np.nan

    ls = LightSource(azdeg=315, altdeg=45)
    rgb = ls.shade(elevation_data, cmap=plt.cm.terrain, vert_exag=1, blend_mode='soft')

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(rgb)
    ax.axis('off')
    plt.savefig(out_image, dpi=200, bbox_inches='tight', pad_inches=0)
    plt.close()
    print(f"[DEBUG] Imagen renderizada en {out_image}")
    return out_image


if __name__ == "__main__":
    # Reemplaza con location_info real de tu pipeline
    location_info = {"state": "Santander", "country": "Colombia"}

    dem_path = get_state_dem(location_info['state'], location_info['country'])
    render_dem(dem_path)

[DEBUG] bbox -> south=5.7067098, north=8.1434194, west=-74.5266563, east=-72.4764662
[DEBUG] descargando DEM para bounds=(-74.5266563, 5.7067098, -72.4764662, 8.1434194) ...
make: Entering directory '/home/wmlegion/.cache/elevation/SRTM1'
curl -s -o spool/N05/N05W075.hgt.gz.temp https://s3.amazonaws.com/elevation-tiles-prod/skadi/N05/N05W075.hgt.gz && mv spool/N05/N05W075.hgt.gz.temp spool/N05/N05W075.hgt.gz
gunzip spool/N05/N05W075.hgt.gz 2>/dev/null || touch spool/N05/N05W075.hgt
gdal_translate -q -co TILED=YES -co COMPRESS=DEFLATE -co ZLEVEL=9 -co PREDICTOR=2 spool/N05/N05W075.hgt cache/N05/N05W075.tif 2>/dev/null || touch cache/N05/N05W075.tif
curl -s -o spool/N06/N06W075.hgt.gz.temp https://s3.amazonaws.com/elevation-tiles-prod/skadi/N06/N06W075.hgt.gz && mv spool/N06/N06W075.hgt.gz.temp spool/N06/N06W075.hgt.gz
gunzip spool/N06/N06W075.hgt.gz 2>/dev/null || touch spool/N06/N06W075.hgt
gdal_translate -q -co TILED=YES -co COMPRESS=DEFLATE -co ZLEVEL=9 -co PREDICTOR=2 spool/N06/N06W

In [12]:
import math
import time
from io import BytesIO
import requests
from PIL import Image


def latlon_to_tile(lat, lon, zoom):
    lat_rad = math.radians(lat)
    n = 2 ** zoom
    x = int((lon + 180.0) / 360.0 * n)
    y = int((1.0 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2.0 * n)
    return x, y


def get_point_map(lat, lon, zoom=17, radius=2, out_path="point_map.png"):
    """
    Descarga un recorte cuadrado de tiles centrado en (lat, lon).
    radius=2 -> grilla de (2*2+1)x(2*2+1) = 5x5 tiles alrededor del centro.
    Sube 'radius' si quieres más contexto alrededor del punto.
    """
    x_center, y_center = latlon_to_tile(lat, lon, zoom)
    x_min, x_max = x_center - radius, x_center + radius
    y_min, y_max = y_center - radius, y_center + radius

    n_tiles = (x_max - x_min + 1) * (y_max - y_min + 1)
    print(f"[DEBUG] centro del tile: {x_center},{y_center} (zoom {zoom})")
    print(f"[DEBUG] descargando {n_tiles} tiles (grilla {2*radius+1}x{2*radius+1})")

    tile_size = 256
    width = (x_max - x_min + 1) * tile_size
    height = (y_max - y_min + 1) * tile_size
    canvas = Image.new("RGB", (width, height))

    headers = {"User-Agent": "agri_land_suitability_pipeline/1.0 (contacto: tu_correo_real@dominio.com)"}

    for x in range(x_min, x_max + 1):
        for y in range(y_min, y_max + 1):
            url = f"https://tile.opentopomap.org/{zoom}/{x}/{y}.png"
            resp = requests.get(url, headers=headers, timeout=10)
            if resp.status_code != 200:
                print(f"[DEBUG] tile {x},{y} falló con status {resp.status_code}")
                time.sleep(0.5)
                continue
            try:
                tile_img = Image.open(BytesIO(resp.content))
                canvas.paste(tile_img, ((x - x_min) * tile_size, (y - y_min) * tile_size))
            except Exception as e:
                print(f"[DEBUG] tile {x},{y} no se pudo abrir: {e}")
            time.sleep(0.5)

    canvas.save(out_path)
    print(f"[DEBUG] Mapa guardado en {out_path} ({width}x{height}px)")
    return out_path


if __name__ == "__main__":
    # Punto de ejemplo que diste
    get_point_map(7.3297, -73.1867, zoom=17, radius=2)

[DEBUG] centro del tile: 38889,62860 (zoom 17)
[DEBUG] descargando 25 tiles (grilla 5x5)
[DEBUG] Mapa guardado en point_map.png (1280x1280px)


In [ ]:
import math
import time
from io import BytesIO
import requests
from PIL import Image


# --- Conversión lat/lon <-> número de tile (fórmula estándar Web Mercator) ---
def latlon_to_tile(lat, lon, zoom):
    lat_rad = math.radians(lat)
    n = 2 ** zoom
    x = int((lon + 180.0) / 360.0 * n)
    y = int((1.0 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2.0 * n)
    return x, y


# --- Descargar y pegar los tiles alrededor de un punto ---
def get_    _map(lat, lon, zoom=17, radius=2, out_path="point_map.png"):
    """
    lat, lon: coordenadas del punto central
    zoom: nivel de zoom (OpenTopoMap soporta hasta 17)
    radius: cuántos tiles agregar alrededor del centro en cada dirección.
            radius=2 -> grilla de 5x5 tiles
    """
    x_center, y_center = latlon_to_tile(lat, lon, zoom)
    x_min, x_max = x_center - radius, x_center + radius
    y_min, y_max = y_center - radius, y_center + radius

    tile_size = 256
    width = (x_max - x_min + 1) * tile_size
    height = (y_max - y_min + 1) * tile_size
    canvas = Image.new("RGB", (width, height))

    headers = {"User-Agent": "agri_land_suitability_pipeline (tu_email@ejemplo.com)"}

    for x in range(x_min, x_max + 1):
        for y in range(y_min, y_max + 1):
            url = f"https://tile.opentopomap.org/{zoom}/{x}/{y}.png"
            resp = requests.get(url, headers=headers, timeout=10)

            if resp.status_code != 200:
                print(f"Tile {x},{y} falló con status {resp.status_code}")
                time.sleep(0.5)
                continue

            try:
                tile_img = Image.open(BytesIO(resp.content))
                canvas.paste(tile_img, ((x - x_min) * tile_size, (y - y_min) * tile_size))
            except Exception as e:
                print(f"Tile {x},{y} falló: {e}")

            time.sleep(0.5)  # respeta el rate limit (~2 req/seg máx)

    canvas.save(out_path)
    print(f"Mapa guardado en {out_path} ({width}x{height}px)")
    return out_path


# Prueba
get_point_map(7.300921,-73.009794, zoom=17, radius=2)

Mapa guardado en point_map.png (1280x1280px)


'point_map.png'

In [14]:
import math
import time
from io import BytesIO
import requests
from PIL import Image, ImageDraw


# --- Conversión lat/lon <-> número de tile (fórmula estándar Web Mercator) ---
def latlon_to_tile(lat, lon, zoom):
    lat_rad = math.radians(lat)
    n = 2 ** zoom
    x = int((lon + 180.0) / 360.0 * n)
    y = int((1.0 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2.0 * n)
    return x, y


def latlon_to_pixel(lat, lon, zoom, tile_size=256):
    """Convierte lat/lon a coordenadas de PIXEL absolutas (no solo tile) en el zoom dado.
    Esto es lo que permite ubicar el punto exacto dentro del canvas, no solo el tile."""
    lat_rad = math.radians(lat)
    n = 2 ** zoom
    x_pixel = (lon + 180.0) / 360.0 * n * tile_size
    y_pixel = (1.0 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2.0 * n * tile_size
    return x_pixel, y_pixel


def draw_marker(canvas, x, y, radius=8, color=(220, 30, 30)):
    """Dibuja un marcador circular con contorno blanco en la posición (x, y) del canvas."""
    draw = ImageDraw.Draw(canvas)
    draw.ellipse(
        [(x - radius, y - radius), (x + radius, y + radius)],
        fill=color, outline=(255, 255, 255), width=3
    )


# --- Descargar y pegar los tiles alrededor de un punto ---
def get_point_map(lat, lon, zoom=17, radius=2, out_path="point_map.png", show_marker=True):
    """
    lat, lon: coordenadas del punto central
    zoom: nivel de zoom (OpenTopoMap soporta hasta 17)
    radius: cuántos tiles agregar alrededor del centro en cada dirección.
            radius=2 -> grilla de 5x5 tiles
    show_marker: si True, dibuja un punto rojo en la ubicación exacta
    """
    x_center, y_center = latlon_to_tile(lat, lon, zoom)
    x_min, x_max = x_center - radius, x_center + radius
    y_min, y_max = y_center - radius, y_center + radius

    tile_size = 256
    width = (x_max - x_min + 1) * tile_size
    height = (y_max - y_min + 1) * tile_size
    canvas = Image.new("RGB", (width, height))

    headers = {"User-Agent": "agri_land_suitability_pipeline (tu_email@ejemplo.com)"}

    for x in range(x_min, x_max + 1):
        for y in range(y_min, y_max + 1):
            url = f"https://tile.opentopomap.org/{zoom}/{x}/{y}.png"
            resp = requests.get(url, headers=headers, timeout=10)

            if resp.status_code != 200:
                print(f"Tile {x},{y} falló con status {resp.status_code}")
                time.sleep(0.5)
                continue

            try:
                tile_img = Image.open(BytesIO(resp.content))
                canvas.paste(tile_img, ((x - x_min) * tile_size, (y - y_min) * tile_size))
            except Exception as e:
                print(f"Tile {x},{y} falló: {e}")

            time.sleep(0.5)  # respeta el rate limit (~2 req/seg máx)

    if show_marker:
        # Pixel absoluto del punto en el mundo, menos el origen del canvas (esquina x_min,y_min)
        px_world, py_world = latlon_to_pixel(lat, lon, zoom, tile_size)
        px_canvas = px_world - x_min * tile_size
        py_canvas = py_world - y_min * tile_size
        draw_marker(canvas, px_canvas, py_canvas)

    canvas.save(out_path)
    print(f"Mapa guardado en {out_path} ({width}x{height}px)")
    return out_path


# Prueba
get_point_map(7.300921,-73.009794, zoom=17, radius=2)

Mapa guardado en point_map.png (1280x1280px)


'point_map.png'

In [2]:
import math
import time
from datetime import datetime
from io import BytesIO
import requests
from PIL import Image, ImageDraw


# --- Conversión lat/lon <-> número de tile (fórmula estándar Web Mercator) ---
def latlon_to_tile(lat, lon, zoom):
    lat_rad = math.radians(lat)
    n = 2 ** zoom
    x = int((lon + 180.0) / 360.0 * n)
    y = int((1.0 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2.0 * n)
    return x, y


def latlon_to_pixel(lat, lon, zoom, tile_size=256):
    """Convierte lat/lon a coordenadas de PIXEL absolutas (no solo tile) en el zoom dado.
    Esto es lo que permite ubicar el punto exacto dentro del canvas, no solo el tile."""
    lat_rad = math.radians(lat)
    n = 2 ** zoom
    x_pixel = (lon + 180.0) / 360.0 * n * tile_size
    y_pixel = (1.0 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2.0 * n * tile_size
    return x_pixel, y_pixel


def draw_marker(canvas, x, y, radius=8, color=(220, 30, 30)):
    """Dibuja un marcador circular con contorno blanco en la posición (x, y) del canvas."""
    draw = ImageDraw.Draw(canvas)
    draw.ellipse(
        [(x - radius, y - radius), (x + radius, y + radius)],
        fill=color, outline=(255, 255, 255), width=3
    )


# --- Descargar y pegar los tiles alrededor de un punto ---
def get_point_map(lat, lon, zoom=17, radius=2, out_prefix="point_map", show_marker=True):
    """
    lat, lon: coordenadas del punto central
    zoom: nivel de zoom (OpenTopoMap soporta hasta 17)
    radius: cuántos tiles agregar alrededor del centro en cada dirección.
            radius=2 -> grilla de 5x5 tiles
    show_marker: si True, dibuja un punto rojo en la ubicación exacta
    out_prefix: nombre base del archivo; se le agrega -vYYMMDDHHMMSS.png automáticamente
    """
    # Timestamp al momento de generar el mapa: YYMMDDHHMMSS (ej: 260803143022)
    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    out_path = f"{out_prefix}-v{timestamp}.png"

    x_center, y_center = latlon_to_tile(lat, lon, zoom)
    x_min, x_max = x_center - radius, x_center + radius
    y_min, y_max = y_center - radius, y_center + radius

    tile_size = 256
    width = (x_max - x_min + 1) * tile_size
    height = (y_max - y_min + 1) * tile_size
    canvas = Image.new("RGB", (width, height))

    headers = {"User-Agent": "agri_land_suitability_pipeline (tu_email@ejemplo.com)"}

    for x in range(x_min, x_max + 1):
        for y in range(y_min, y_max + 1):
            url = f"https://tile.opentopomap.org/{zoom}/{x}/{y}.png"
            resp = requests.get(url, headers=headers, timeout=10)

            if resp.status_code != 200:
                print(f"Tile {x},{y} falló con status {resp.status_code}")
                time.sleep(0.5)
                continue

            try:
                tile_img = Image.open(BytesIO(resp.content))
                canvas.paste(tile_img, ((x - x_min) * tile_size, (y - y_min) * tile_size))
            except Exception as e:
                print(f"Tile {x},{y} falló: {e}")

            time.sleep(0.5)  # respeta el rate limit (~2 req/seg máx)

    if show_marker:
        # Pixel absoluto del punto en el mundo, menos el origen del canvas (esquina x_min,y_min)
        px_world, py_world = latlon_to_pixel(lat, lon, zoom, tile_size)
        px_canvas = px_world - x_min * tile_size
        py_canvas = py_world - y_min * tile_size
        draw_marker(canvas, px_canvas, py_canvas)

    canvas.save(out_path)
    print(f"Mapa guardado en {out_path} ({width}x{height}px)")
    return out_path


# Prueba
get_point_map(3.580109040361371, -76.31299479308868, zoom=17, radius=2)

Mapa guardado en point_map-v260803184352.png (1280x1280px)


'point_map-v260803184352.png'

In [3]:
import ee

# Inicializar Earth Engine (asegúrate de haber ejecutado ee.Authenticate() previamente si es necesario)
ee.Initialize()

def get_country_boundary_from_coords(lat, lon):
    """
    Toma latitud y longitud, busca de forma espacial el país y estado 
    correspondientes en la base de datos FAO GAUL de Earth Engine, 
    y retorna tanto sus metadatos de texto como sus geometrías vectoriales.
    """
    # 1. Crear el punto geográfico con las coordenadas de entrada (Nota: GEE usa [longitud, latitud])
    point = ee.Geometry.Point([lon, lat])
    
    # 2. Cargar las colecciones oficiales de FAO GAUL (Nivel 0: País, Nivel 1: Estado/Provincia)
    gaul_level0 = ee.FeatureCollection("FAO/GAUL/2015/level0")
    gaul_level1 = ee.FeatureCollection("FAO/GAUL/2015/level1")
    
    # 3. Filtrar de forma espacial el polígono que contiene el punto de interés
    country_feature = gaul_level0.filterBounds(point).first()
    state_feature = gaul_level1.filterBounds(point).first()
    
    # 4. Extraer los nombres oficiales en texto (usando .getInfo() para consulta puntual)
    country_name = country_feature.get('ADM0_NAME').getInfo()
    state_name = state_feature.get('ADM1_NAME').getInfo()
    
    print(f"--- Diagnóstico Geográfico ---")
    print(f"Coordenadas consultadas: Lat: {lat}, Lon: {lon}")
    print(f"País detectado: {country_name}")
    print(f"Estado/Provincia detectado: {state_name}")
    print(f"------------------------------")
    
    # 5. Retornar los metadatos y los objetos geométricos nativos para el pipeline
    return {
        "country_name": country_name,
        "state_name": state_name,
        "country_polygon": country_feature.geometry(),
        "state_polygon": state_feature.geometry()
    }

# ==========================================
# Ejemplo de prueba con tus coordenadas
# ==========================================
if __name__ == "__main__":
    # Probamos con las coordenadas de ejemplo (Australia)
    lat_test = -24.8660
    lon_test = 152.3489
    
    result = get_country_boundary_from_coords(lat_test, lon_test)
    
    # Puedes usar 'result["country_polygon"]' directamente en otras funciones de GEE,
    # por ejemplo, para enmascarar rásters, recortar mapas o calcular estadísticas zonales.

--- Diagnóstico Geográfico ---
Coordenadas consultadas: Lat: -24.866, Lon: 152.3489
País detectado: Australia
Estado/Provincia detectado: Queensland
------------------------------


In [7]:
import ee
import urllib.request

# 1. Inicializar Earth Engine
ee.Initialize()

def download_boundary_as_png(lat, lon, output_filename="map_boundary.png", region_type="state"):
    """
    Extrae la geometría usando coordenadas, la rasteriza como FeatureCollection 
    y genera un enlace de descarga directa en formato PNG hacia la PC local.
    """
    # Crear el punto de referencia
    point = ee.Geometry.Point([lon, lat])
    
    # Seleccionar la colección (Nivel 0: País, Nivel 1: Estado/Provincia)
    if region_type == "country":
        feature = ee.FeatureCollection("FAO/GAUL/2015/level0").filterBounds(point).first()
    else:
        feature = ee.FeatureCollection("FAO/GAUL/2015/level1").filterBounds(point).first()
        
    name_key = 'ADM0_NAME' if region_type == "country" else 'ADM1_NAME'
    region_name = feature.get(name_key).getInfo()
    print(f"Zona detectada: {region_name}")
    
    # Obtener la geometría exacta para recortar la vista del mapa
    geom = feature.geometry()
    
    # SOLUCIÓN: Envolver el 'feature' en una FeatureCollection para que .paint() no falle
    feature_collection = ee.FeatureCollection([feature])
    
    # Convertir el vector a una imagen binaria (1 dentro del polígono, 0 afuera)
    image = ee.Image().byte().paint(feature_collection, 1).visualize(
        palette=['2e7d32'] # Verde oscuro institucional
    )
    
    # Generar la URL de descarga de la imagen PNG para la región exacta del polígono
    thumbnail_url = image.getThumbURL({
        'region': geom,
        'dimensions': 1024, # Resolución de la imagen en píxeles (ancho máximo)
        'format': 'png'
    })
    
    print(f"Generando imagen PNG para {region_name}...")
    
    # Descargar la imagen directamente a tu PC
    urllib.request.urlretrieve(thumbnail_url, output_filename)
    
    print(f"¡Éxito! El mapa en PNG se ha guardado en tu PC como: '{output_filename}'")

# ==========================================
# Ejecutar la prueba corregida
# ==========================================
if __name__ == "__main__":
    lat_test = 7.4584221918243045
    lon_test = -73.222052853104
    
    # Descarga el estado de Queensland como imagen PNG sin errores
    download_boundary_as_png(lat_test, lon_test, output_filename="Greenland_Region_Map.png", region_type="state")

Zona detectada: Santander
Generando imagen PNG para Santander...
¡Éxito! El mapa en PNG se ha guardado en tu PC como: 'Greenland_Region_Map.png'


In [8]:
import ee
import urllib.request

# 1. Inicializar Earth Engine
ee.Initialize()

def download_real_satellite_image_png(lat, lon, output_filename="satellite_real_map.png", region_type="state"):
    """
    Extrae la geometría por coordenadas, carga el mosaico satelital real (Sentinel-2 RGB),
    lo recorta a los límites de la región y lo descarga como una imagen PNG real.
    """
    # Crear el punto de referencia
    point = ee.Geometry.Point([lon, lat])
    
    # Seleccionar la colección administrativa (Nivel 0: País, Nivel 1: Estado/Provincia)
    if region_type == "country":
        feature = ee.FeatureCollection("FAO/GAUL/2015/level0").filterBounds(point).first()
    else:
        feature = ee.FeatureCollection("FAO/GAUL/2015/level1").filterBounds(point).first()
        
    name_key = 'ADM0_NAME' if region_type == "country" else 'ADM1_NAME'
    region_name = feature.get(name_key).getInfo()
    print(f"Zona detectada: {region_name}")
    
    geom = feature.geometry()
    
    # Cargar la colección de Sentinel-2 para obtener una imagen libre de nubes (True Color RGB)
    # Bandas: B4 (Rojo), B3 (Verde), B2 (Azul)
    s2_collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(geom)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
        .select(['B4', 'B3', 'B2'])
    )
    
    # Obtener una imagen compuesta limpia mediante la mediana (elimina nubes y sombras temporales)
    composite_image = s2_collection.median().clip(geom)
    
    # Parámetros de visualización para color real (ajuste de brillo y contraste con min/max)
    vis_params = {
        'min': 0,
        'max': 3000,
        'bands': ['B4', 'B3', 'B2'] # RGB
    }
    
    # Aplicar la visualización para convertirla en imagen de 8 bits lista para PNG
    visualized_image = composite_image.visualize(**vis_params)
    
    # Generar la URL de descarga del mapa satelital real
    thumbnail_url = visualized_image.getThumbURL({
        'region': geom,
        'dimensions': 1024, # Resolución en píxeles (ancho máximo)
        'format': 'png'
    })
    
    print(f"Generando imagen satelital real en color para {region_name}...")
    
    # Descargar la imagen a la PC
    urllib.request.urlretrieve(thumbnail_url, output_filename)
    
    print(f"¡Éxito! El mapa satelital real se ha guardado como: '{output_filename}'")

# ==========================================
# Ejecutar la descarga del mapa satelital real
# ==========================================
if __name__ == "__main__":
    lat_test = -24.8660
    lon_test = 152.3489
    
    # Descarga la imagen real a color de Queensland, Australia
    download_real_satellite_image_png(lat_test, lon_test, output_filename="Queensland_Satellite_Real.png", region_type="state")

Zona detectada: Queensland
Generando imagen satelital real en color para Queensland...


KeyboardInterrupt: 

In [14]:
import ee
import urllib.request

# 1. Inicializar Earth Engine
ee.Initialize()

def download_quick_overview_map(lat, lon, output_filename="country_overview.png", region_type="state"):
    """
    Descarga una imagen satelital general y rápida de la región 
    utilizando MODIS de forma ligera y sin errores de bandas.
    """
    # Crear el punto de referencia
    point = ee.Geometry.Point([lon, lat])
    
    # Seleccionar la colección administrativa
    if region_type == "country":
        feature = ee.FeatureCollection("FAO/GAUL/2015/level0").filterBounds(point).first()
    else:
        feature = ee.FeatureCollection("FAO/GAUL/2015/level1").filterBounds(point).first()
        
    name_key = 'ADM0_NAME' if region_type == "country" else 'ADM1_NAME'
    region_name = feature.get(name_key).getInfo()
    print(f"Zona detectada: {region_name}")
    
    geom = feature.geometry()
    
    # Cargar la imagen MODIS estática
    image = ee.Image('MODIS/006/MCD43A4/2020_01_01')
    
    # Recortar al límite geográfico
    clipped = image.clip(geom)
    
    # Especificar las bandas RGB directamente dentro de la visualización
    vis_params = {
        'bands': ['Nadir_Reflectance_Band1', 'Nadir_Reflectance_Band4', 'Nadir_Reflectance_Band3'],
        'min': 0,
        'max': 3000
    }
    
    vis_image = clipped.visualize(**vis_params)
    
    # Reducir las dimensiones a 400 píxeles para que la descarga sea ultrarrápida
    thumbnail_url = vis_image.getThumbURL({
        'region': geom,
        'dimensions': 1024, 
        'format': 'png'
    })
    
    print(f"Descargando imagen general rápida para {region_name}...")
    
    # Descargar directamente a la PC
    urllib.request.urlretrieve(thumbnail_url, output_filename)
    
    print(f"¡Listo! Imagen guardada como: '{output_filename}'")

# ==========================================
# Ejecutar la prueba corregida
# ==========================================
if __name__ == "__main__":
    lat_test = 7.4584221918243045
    lon_test = -73.222052853104
    
    download_quick_overview_map(lat_test, lon_test, output_filename="Queensland_Overview.png", region_type="state")

Zona detectada: Santander
Descargando imagen general rápida para Santander...
¡Listo! Imagen guardada como: 'Queensland_Overview.png'


In [22]:
import ee
import urllib.request

# 1. Inicializar Earth Engine
ee.Initialize()

def download_clean_overview_map(lat, lon, output_filename="country_overview_clean.png", region_type="state"):
    """
    Descarga una imagen panorámica rápida de la región usando un mosaico 
    anual de MODIS para evitar huecos vacíos.
    """
    # Crear el punto de referencia
    point = ee.Geometry.Point([lon, lat])
    
    # Seleccionar la colección administrativa
    if region_type == "country":
        feature = ee.FeatureCollection("FAO/GAUL/2015/level0").filterBounds(point).first()
    else:
        feature = ee.FeatureCollection("FAO/GAUL/2015/level1").filterBounds(point).first()
        
    name_key = 'ADM0_NAME' if region_type == "country" else 'ADM1_NAME'
    region_name = feature.get(name_key).getInfo()
    print(f"Zona detectada: {region_name}")
    
    geom = feature.geometry()
    
    # En lugar de una sola fecha fija, usamos un año completo (2020) 
    # y aplicamos .median() para rellenar automáticamente cualquier espacio vacío.
    collection = ee.ImageCollection('MODIS/006/MCD43A4').filterDate('2020-01-01', '2020-12-31')
    image = collection.median()
    
    # Recortar al límite geográfico
    clipped = image.clip(geom)
    
    # Parámetros de visualización RGB
    vis_params = {
        'bands': ['Nadir_Reflectance_Band1', 'Nadir_Reflectance_Band4', 'Nadir_Reflectance_Band3'],
        'min': 0,
        'max': 3000
    }
    
    vis_image = clipped.visualize(**vis_params)
    
    # Generar la URL de descarga de la imagen rápida
    thumbnail_url = vis_image.getThumbURL({
        'region': geom,
        'dimensions': 512, 
        'format': 'png'
    })
    
    print(f"Descargando imagen general limpia para {region_name}...")
    
    # Descargar directamente a la PC
    urllib.request.urlretrieve(thumbnail_url, output_filename)
    
    print(f"¡Listo! Imagen limpia guardada como: '{output_filename}'")

# ==========================================
# Ejecutar la prueba con el mapa limpio
# ==========================================
if __name__ == "__main__":
    # lat_test = 7.4584221918243045
    # lon_test = -73.222052853104
    
    lat_test = -24.8660
    lon_test = 152.3489
    
    download_clean_overview_map(lat_test, lon_test, output_filename="Queensland_Overview_Clean.png", region_type="state")

Zona detectada: Queensland
Descargando imagen general limpia para Queensland...
¡Listo! Imagen limpia guardada como: 'Queensland_Overview_Clean.png'


In [2]:
import ee
import urllib.request

# Inicializar Earth Engine
ee.Initialize()

def download_regional_map(lat, lon, output_filename="map_output.png", region_type="state", map_type="real"):
    """
    Descarga un mapa general de la región según el tipo seleccionado:
    - map_type="real": Imagen satelital real a color (compuesto anual MODIS limpio).
    - map_type="elevation": Mapa de altitudes (DEM) con paleta de colores topográfica.
    """
    # 1. Crear el punto de referencia y extraer la geometría administrativa
    point = ee.Geometry.Point([lon, lat])
    
    if region_type == "country":
        feature = ee.FeatureCollection("FAO/GAUL/2015/level0").filterBounds(point).first()
    else:
        feature = ee.FeatureCollection("FAO/GAUL/2015/level1").filterBounds(point).first()
        
    name_key = 'ADM0_NAME' if region_type == "country" else 'ADM1_NAME'
    region_name = feature.get(name_key).getInfo()
    geom = feature.geometry()
    
    print(f"Zona detectada: {region_name} | Modo de mapa: {map_type.upper()}")
    
    # 2. Seleccionar la fuente de datos según el tipo de mapa pedido
    if map_type == "elevation":
        # Usar el modelo de elevación digital (DEM) de Copernicus / SRTM
        image = ee.Image('USGS/SRTMGL1_003').select('elevation')
        clipped = image.clip(geom)
        
        # Paleta topográfica clásica (bajos/valles en verde, medios en amarillo/marrón, cumbres en blanco)
        vis_params = {
            'min': 0,
            'max': 4000, # Rango amplio para zonas montañosas
            'palette': [
                '004400', # 0 – 364 m: Valles profundos, tierras bajas o planicies costeras
                '006600', # 364 – 727 m: Tierras bajas / Bosques húmedos tropicales
                '38a800', # 727 – 1,091 m: Transición baja / Inicio de piedemontes
                '73d216', # 1,091 – 1,455 m: Zonas cafeteras o agrícolas de clima templado
                'b2d235', # 1,455 – 1,818 m: Laderas de montaña media
                'fce94f', # 1,818 – 2,182 m: Tierras medias altas / Bosques andinos
                'e9b96e', # 2,182 – 2,545 m: Zonas altoandinas / Transición fría
                'c87d32', # 2,545 – 2,909 m: Páramos bajos / Montaña alta
                '8f5902', # 2,909 – 3,273 m: Páramos altos / Suelos rocosos fríos
                '5c3a21', # 3,273 – 3,636 m: Superpáramo / Cumbres escarpadas
                '8b8b8b'  # 3,636 – 4,000+ m: Picos más altos / Gris roca alpino
            ]
        }
        
    else:
        # Modo por defecto: Imagen real panorámica limpia (Compuesto anual MODIS)
        collection = ee.ImageCollection('MODIS/006/MCD43A4').filterDate('2020-01-01', '2020-12-31')
        image = collection.median()
        clipped = image.clip(geom)
        
        vis_params = {
            'bands': ['Nadir_Reflectance_Band1', 'Nadir_Reflectance_Band4', 'Nadir_Reflectance_Band3'],
            'min': 0,
            'max': 3000
        }
        
    # 3. Aplicar visualización y generar URL de descarga rápida
    vis_image = clipped.visualize(**vis_params)
    
    thumbnail_url = vis_image.getThumbURL({
        'region': geom,
        'dimensions': 500, # Resolución rápida y ligera para fotos de referencia
        'format': 'png'
    })
    
    print(f"Generando y descargando el archivo '{output_filename}'...")
    urllib.request.urlretrieve(thumbnail_url, output_filename)
    print(f"¡Éxito! Mapa guardado correctamente.")

# ==========================================
# Ejemplos de uso:
# ==========================================
if __name__ == "__main__":
    lat_test = -24.8660
    lon_test = 152.3489
    
    # lat_test = 7.4584221918243045
    # lon_test = -73.222052853104

    # Opción A: Descargar la vista satelital real limpia
    # download_regional_map(lat_test, lon_test, output_filename="mapa_Real.png", region_type="state", map_type="real")
    
    # Opción B: Descargar el mapa de altitudes (topografía)
    download_regional_map(lat_test, lon_test, output_filename="mapa_Elevation_sant_04.png", region_type="state", map_type="elevation")

Zona detectada: Queensland | Modo de mapa: ELEVATION
Generando y descargando el archivo 'mapa_Elevation_sant_04.png'...
¡Éxito! Mapa guardado correctamente.


In [6]:
from datetime import datetime
from pathlib import Path
import requests
import ee


# Paleta topográfica: valles/tierras bajas en verde, zonas medias en
# amarillo/marrón, cumbres en blanco/gris (referencia orientativa de rangos
# altitudinales para Colombia, ajustar 'max' si tu región es más alta/baja)
ELEVATION_PALETTE = [
    '004400',  # 0 – 364 m: Valles profundos, tierras bajas o planicies costeras
    '006600',  # 364 – 727 m: Tierras bajas / Bosques húmedos tropicales
    '38a800',  # 727 – 1,091 m: Transición baja / Inicio de piedemontes
    '73d216',  # 1,091 – 1,455 m: Zonas cafeteras o agrícolas de clima templado
    'b2d235',  # 1,455 – 1,818 m: Laderas de montaña media
    'fce94f',  # 1,818 – 2,182 m: Tierras medias altas / Bosques andinos
    'e9b96e',  # 2,182 – 2,545 m: Zonas altoandinas / Transición fría
    'c87d32',  # 2,545 – 2,909 m: Páramos bajos / Montaña alta
    '8f5902',  # 2,909 – 3,273 m: Páramos altos / Suelos rocosos fríos
    '5c3a21',  # 3,273 – 3,636 m: Superpáramo / Cumbres escarpadas
    '8b8b8b',  # 3,636 – 4,000+ m: Picos más altos / Gris roca alpino
]


def download_regional_elevation_map(lat, lon, region_type="state", out_prefix="regional_elevation",
                                     output_dir="../img/maps", dimensions=720,
                                     min_elevation=0, max_elevation=4000):
    """
    Descarga un mapa de elevación (DEM, paleta topográfica) de la región
    administrativa (state o country) que contiene el punto dado.

    lat, lon: coordenadas del punto de referencia (se usa solo para detectar
               la región administrativa, no delimita el recorte)
    region_type: "state" (nivel 1, departamento/provincia) o "country" (nivel 0)
    dimensions: tamaño en pixeles del lado más largo de la imagen
    min_elevation, max_elevation: rango de la paleta de colores (metros)
    """
    point = ee.Geometry.Point([lon, lat])

    if region_type == "country":
        feature = ee.FeatureCollection("FAO/GAUL/2015/level0").filterBounds(point).first()
        name_key = 'ADM0_NAME'
    else:
        feature = ee.FeatureCollection("FAO/GAUL/2015/level1").filterBounds(point).first()
        name_key = 'ADM1_NAME'

    region_name = feature.get(name_key).getInfo()
    geom = feature.geometry()
    print(f"[DEBUG] Zona detectada: {region_name}")

    image = ee.Image('USGS/SRTMGL1_003').select('elevation')
    clipped = image.clip(geom)

    vis_image = clipped.visualize(
        min=min_elevation,
        max=max_elevation,
        palette=ELEVATION_PALETTE,
    )

    thumbnail_url = vis_image.getThumbURL({
        'region': geom,
        'dimensions': dimensions,
        'format': 'png',
    })

    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    filename = f"{out_prefix}-v{timestamp}.png"
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    out_path = output_path / filename

    resp = requests.get(thumbnail_url, timeout=30)
    resp.raise_for_status()
    out_path.write_bytes(resp.content)

    print(f"Imagen guardada en {out_path}")
    return out_path


if __name__ == "__main__":
    lat_test = -24.8660
    lon_test = 152.3489
    
    # lat_test = 7.4584221918243045
    # lon_test = -73.222052853104

    download_regional_elevation_map(lat_test, lon_test, region_type="state")

[DEBUG] Zona detectada: Queensland
Imagen guardada en ../img/maps/regional_elevation-v260805193354.png
